# EATS PL Report - Telescopes
Template by André Moitinho

***

### Student number: 56669
### Name: André Cardoso

***

<div class="alert alert-block alert-info">

## Instructions

**Read carefully**. Failing to follow instructions will impact on the final grade.

* Fill in your name and number above
* This template provides blocks of instructions that guide you through the activity and report. Put your codes and comments below the corresponding blocks. 
* Add as many code and markdown cells as necessary. 
* The blue background colour in the markdown cells (like this one) is reserved for the instruction blocks. Use white cells for your comments.
* Write a clear report:
    * Provide justifications and comments.
    * Figures and labels (legends, axes, etc) should be clear and easy to read.
    * Tables should be neatly formatted and include labels.

**To submit your report:**
    
* Clean up and make sure all works well: In the "Kernel" menu select "Restart & Run All" (or use the "fast forward/double arrow" icon). Wait until the process is completed
* Save to a pdf file:
    * JupyterNotebook users: In the "File" menu -> "Download As" -> "**PDF via HTML (.html)**". 
    * JupyterLab users: In the "File" menu -> "Save and Export Notebook As" -> "**Webpdf**".
    * If you don't see the Webpdf option, you'll need to install the _nbconvert[webpdf]_ package, as explained in the _Software and data installation instructions lab activities_ link in the course moodle.
* Check that the PDF file is fine (no truncated lines, nothing missing, etc)
* Name the file: NumberFirstnameLastname.pdf* (e.g. 10101ClaudeShannon.pdf)
* Upload the pdf file to the "PL Report - Telescopes" assignement area. 

### Tips
* Run a cell  - keyboard shortcut: shift-enter
* To create "markdown" cells like this one, For comments, not for code: create a new cell. In the menu above to the right of the "fast-forward" icon there's a dropdown button with the default value "Code". Change to "Markdown".
* To display an image from a file on disk (e.g. image.png): In a **markdown cell** you can use *\!\[some text\](image.png)*, or simply drag and drop!
</div>

---

<div class="alert alert-block alert-info">  

## Setup
Put all your imports (e.g. numpy,matplotlib, etc)  in the cell bellow
</div>


In [6]:
from pathlib import Path

import numpy as np

import pandas as pd

from astropy.io import fits
from astropy.nddata import CCDData
import ccdproc
from ccdproc import ImageFileCollection

<div class="alert alert-block alert-info">
    
## <center>PART 1 -  KNOWING YOUR DATA</center>

    
### Basic image information
    
For the set of fits files in the *0_mainData* folder, create a table with: image_name, filter exposure_time, imagetype, object_name.  Show:
* the command line you used for this 
* the table (use the “pandas” package and display the column names at the top)
</div>


Primeiramente cria-se uma variável Path para definir um caminho de leitura e
cria-se uma image file collection que facilita o manuseamento dos ficheiros.

Para ter a certeza das keywords usadas no header dos ficheiros faz-se um print do header de uma das imagens

In [2]:
read_path = Path("ccdDataETS/0_mainData")
imcol = ImageFileCollection(read_path)

print(next(imcol.hdus()).header)

SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                   16 / array data type                                NAXIS   =                    2 / number of array dimensions                     NAXIS1  =                 1072                                                  NAXIS2  =                 1024                                                  ORIGIN  = 'Copyright (C) 1991-1998 GKR Computer Consulting' / FITS file originatDATE    = '2001-06-06T00:19:27' / Date FITS file was generated                  IRAF-TLM= '15:49:40 (05/06/2001)' / Time of last modification                   EXPTIME =                    5                                                  FILTER  = 'I       '                                                            IMAGETYP= 'OBJECT  '                                                            CCDSUM  = '1 1     '                                                            CCDSEC  = '[1:1024,1:1024]'             

Cria-se uma tabela apenas com os atributos pretendidos e
convertendo-a para um DataFrame do pandas para uma formatação mais limpa

In [3]:
table = imcol.summary['file', 'filter', 'exptime', 'imagetyp', 'object']

df = table.to_pandas()
df.columns = ['File', 'Filter', 'Exposure Time', 'Image Type', 'Object']
display(df)

,File,Filter,Exposure Time,Image Type,Object
0,3241o.fits,I,5,OBJECT,NGC2420
1,3249o.fits,I,90,OBJECT,NGC2420
2,3262o.fits,V,10,OBJECT,NGC2420
3,3265o.fits,V,180,OBJECT,NGC2420
4,4015b.fits,UNKNOWN,1,BIAS,UNKNOWN
5,4016b.fits,UNKNOWN,1,BIAS,UNKNOWN
6,4017b.fits,UNKNOWN,1,BIAS,UNKNOWN
7,4075f.fits,V,3,FLAT,Twilight
8,4076f.fits,V,3,FLAT,Twilight
9,4078f.fits,V,3,FLAT,Twilight


<div class="alert alert-block alert-info">

### Determine the CCD gain and readout noise:
    
For this, follow the recipe given in Howell p.71-73.
    
In the *gainNoise* folder, use images bias1, bias2, domeV1 and domeV2 to determine the readout noise and gain of the CCD. Don't forget to select an appropriate section of the image for the calculations (don’t use the overscan section).  
Update the CCD header of your images with the values you have just determined.
    
Verify in one image header that the values have indeed been updated. 

* Show the updated header

    



</div>


In [21]:
gainNoise_path = Path("ccdDataETS/gainNoise")

dome = CCDData.read(gainNoise_path / "domeV1.fits",unit="adu")
print(dome.header.get("trimsec"))

[1:1024,1:1024]


In [31]:
#Calculo do ganho
#-------------------------
#abrir os fits

def loaddata(file):
    return fits.getdata(gainNoise_path / file)[1:1024,1:1024].astype(float)

b1 = loaddata("bias1.fits")
b2 = loaddata("bias2.fits")
dV1 = loaddata("domeV1.fits")
dV2 = loaddata("domeV2.fits")

#calculo dos desvios padrao dos fits subtraídos
stddev_b = np.std(np.subtract(b1, b2))
stddev_d = np.std(np.subtract(dV1, dV2))

#calculo do numerador e divisor calculando as médias diretamente
num = (np.mean(dV1)+np.mean(dV2)) - (np.mean(b1)+np.mean(b2))
div = np.square(stddev_d) - np.square(stddev_b)

#Ganho
gain = num/div

#Calculo do readout noise
#-------------------------
readout = (gain*stddev_b)/np.sqrt(2)

print("ganho =",gain)
print("readout noise = "+str(readout)+"e-")

ganho = 1.2795611780413165
readout noise = 6.219506789564265e-


<div class="alert alert-block alert-info">
   
### Tracking the bias level - I
    
Display one of the bias images in DS9. Remember, they are also called “zero” images. You can identify them by checking the list created above.

In the “scale” menu on the top bar, uncheck the “use DATASEC” option. You will now be able to also see the overscan strip. 
    
Interactively, roughly estimate by eye the mean level of the data section and of the overscan section (explore the pixel table and smooth tools offered in the “analysis” menu on the top bar). 
    
* What values do you estimate?
* Show a snapshot of your desktop with the DS9 main window and the auxiliary tools panel(s) used in this analysis.

</div>


<div class="alert alert-block alert-info">

### Tracking the bias level - II
    
Now, from to the *superbias* folder. Consider appropriate sections (avoid obvious defects) of all images to compute the necessary statistics for:
* Making a scatter plot with x= image number (e.g. 5239 for image 5239o.fits) and y= mean bias level
* Making a scatter plot with x= image number and y= mean bias - mean overscan
* Place both plots side-by-side in a same figure
* What do you conclude?

Because the bias images are approximately uniform, the standard deviation of the counts in a small box (e.g. 5x5 or 10x10 pixels) give an idea of the read out noise (here in ADUs, not electrons).  From one of your bias images, compute the typical standard deviation in the Data Section and in the Overscan Section. Take 3 boxes spread out each of these sections to estimate the typical values.
* What do you conclude?


    
</div>


<div class="alert alert-block alert-info">

## <center> PART 2 - DATA PROCESSING </center>


### Overscan and Trim

Select appropriate overscan and trim areas. 
* Justify your selection. 

Apply the overscan correction and trim all the images. in the *0_mainData* folder. Save the processed images in another folder (e.g. 1_processed). 

Verify that the images have been processed:
* Show the header of one of the newly corrected images.
* Present a table showing that all images have been processed.


</div>


<div class="alert alert-block alert-info">

### Creating the Bias correction image

From the *superbias* folder, apply the overscan correction and trim all the 100 images.

Combine them to create the master bias correction image (we’ll refer to it as the superZero image). 
* Explain your choices for the image combination (combine method, parameters, etc)

Estimate the typical mean level and standard deviation (noise) of the superZero image.
* How does it compare with the values from any of the individual images used to produce it? Why?
* Display a picture of superZero.    

</div>


<div class="alert alert-block alert-info">

### Bias correction

In the processed folder, use the superZero image to apply the bias correction to the images that should be corrected. Make sure a keyword to was added to flag that the images have been bias subtracted.

Verify that the images have been processed:
* Show the header of one of the newly corrected images.
* Present a table showing that all images have been processed.    

</div>


<div class="alert alert-block alert-info">

### Creating the flatfield correction images

Create the flatfield correction images for both filters. Use the task default parameters. What problem did you find? Fix it! 
* How did you fix the problem? (NOTE: take care not to use the combined flat images you've created in the previous attempt when generating the new good ones)
* Show the good and the bad flatfield correction images you have created. Identify the images clearly with labels.     

</div>


<div class="alert alert-block alert-info">

### Flatfield correction

Apply the flatfield correction to the object images. 
Verify that the images have been processed:
* Show the header of one of the newly corrected images.
* Present a table showing that all images have been processed.    

</div>


<div class="alert alert-block alert-info">
    
You did it! The final science images are ready for delivery to the client. You have processed and removed the detector footprint down to the percent level.
* Display the 4 final  corrected images. Identify each images clearly with labels 

</div>
